# falseref demo

Checks whether the references in a document exist and are cited correctly. It uses rules only, with no LLM.

1. **Read** the reference list. Word files are converted to PDF; entries are cut from the page layout and parsed by GROBID.
2. **Look up** every identifier the reference gives: the DOI (Crossref / doi.org), the URL (the live page and its `<meta>` tags), and the title (Crossref + OpenAlex search).
3. **Decide** a tier: does the cited work exist, and is it cited correctly?

| tier | meaning |
|---|---|
| ✅ Verified | the work was found and the citation matches it |
| ⚠️ Exists, citation errors | the work was found, but a field is wrong (title, author, year, volume, DOI, ...) |
| 🚩 No match found | nothing shows the work exists, or a cited DOI/URL is dead or belongs to another work |
| ❓ Inconclusive | a lookup was blocked or failed, or it is a book that may be reprinted |
| 🤖 GenAI tool | output of a generative AI tool, listed but not checked |

**Needs:** `pip install -e .` from this folder, a GROBID server for documents (default `http://localhost:8070`, or set `GROBID_URL`), and LibreOffice (`soffice`) for .doc/.docx.

In [ ]:
import logging

import pandas as pd
import requests

from falseref import check_path, check_text, http_status_counts, summary, to_frame, write_csv
from falseref.references.grobid import GROBID_URL

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
pd.set_option("display.max_colwidth", 80)

try:
    print("GROBID", requests.get(f"{GROBID_URL}/api/version", timeout=5).json())
except requests.RequestException:
    print(f"GROBID not reachable at {GROBID_URL}: documents fall back to the APA regex parser")

## 1. A pasted reference list

Entries are separated by blank lines and parsed with the APA regex parser, so this part needs no GROBID.

In [ ]:
REFERENCE_LIST = """
European Banking Authority. (2021). Report on machine learning in credit risk. https://www.eba.europa.eu/sites/default/documents/files/document_library/Publications/Reports/ 2021/1015487/Report%20on%20ML%20in%20credit%20risk.pdf

Kumar, A., & Ravi, V. (2016). A survey of the applications of machine learning in credit risk assessment. Computers & Operations Research, 57, 47-66. https://doi.org/10.1016/j.cor.2014.12.005

Fraunhofer Institute. (2023). AI bias in power systems: Predictive maintenance case studies. https://publica.fraunhofer.de/entities/publication/9a7e3f4c-3f3d-4c6d-8c9a-
1e4b3c5a4d2f

IBM. (2024). The role of AI in predictive maintenance. https://www.ibm.com/blog/the-role-of-ai-in-predictive-maintenance

McKinsey Global Institute. (2020). How will automation affect jobs? https://www.mckinsey.com/featured-insights/future-of-work/how-will-automation-
Affect-job

Nature Machine Intelligence. (2019). The ethical challenges of black-box AI. https://www.nature.com/articles/s42256-019-0138-9

Neural Concept. (2024). Predictive maintenance machine learning: A practical guide. https://neuralconcept.com/blog/predictive-maintenance-machine-learning-guide

OECD. (2021). Liability for AI and emerging technologies. https://www.oecd.org/digital/ai/liability-ai/

Royal Society. (2020). Public views on machine learning. https://royalsociety.org/topics-policy/projects/machine-learning/public-views

Siemens. (2024). Senseye predictive maintenance. https://www.siemens.com/global/en/products/services/senseye-cloud-application.html

Springer Nature. (2025). A comprehensive review on AI-driven predictive maintenance in vehicles. https://link.springer.com/article/10.1007/s42452-025-01587-3

Stanford Human-Centered AI. (2021). Anthropomorphizing AI can be dangerous. https://hai.stanford.edu/news/anthropomorphizing-ai-can-be-dangerous

AACA. (2023). Digital competence and emerging technologies in Australian architecture. Architects Accreditation Council of Australia. https://aaca.org.au

AACA. (2023). National Standard of Competency for Architects. Architects Accreditation Council of Australia. https://aaca.org.au

Daly, A., Mann, M., & Devitt, S. (2023). Good data: Data justice, AI governance and the future of society. Australian National University Press. https://press.anu.edu.au
"""

In [ ]:
pasted = check_text(REFERENCE_LIST)
pasted_table = to_frame([pasted])
pasted_table[["n", "tier", "cited_title", "sources", "issues", "notes"]]

## 2. Inside one reference

Each step is its own module, so one reference can be followed through the pipeline:
`references` (parse) → `sources` (DOI, URL and title lookups) → `verdict` (sum the records, pick the tier).

In [ ]:
from falseref.references.identifiers import cited_links
from falseref.sources import doi_source, title_source, url_source

ref = pasted.references[1]  # Kumar & Ravi (2016): its DOI belongs to another paper
print(ref, "\n")

doi, url = cited_links(ref)
for source in (doi_source(doi), url_source(url, ref.title), title_source(ref, url)):
    print(f"{source['name']:6} {source['state']:12} {source.get('note') or ''}")

In [ ]:
result = pasted.results[1]
print(result.label)
for issue in result.issues:
    print(f"  {issue.severity:5} {issue.field:8} {issue.message}")
for note in result.notes:
    print("  note:", note)

## 3. A batch of submissions

A single PDF/DOC/DOCX, or a zip of them. The report has one row per reference:
`cited_*` is what the reference says, `found_*` is what the sources say, and `check_*` is the per-field status
(ok / minor / major / not confirmed / no data / not checked). `issues` lists every discrepancy, and `notes` lists what could not be confirmed.

In [ ]:
SUBMISSIONS = "../Re__Referencing_Mistakes.zip"

reports = check_path(SUBMISSIONS)
batch = to_frame(reports)
write_csv(batch, "reference_report.csv")
batch[["file", "n", "tier", "cited_title", "sources", "issues", "notes"]]

In [ ]:
print(summary(reports))
print()
print(http_status_counts(batch).to_string())